
# Lab Assignment 4: NLP Preprocessing and Text Classification

**Name:** Anushka Sunil Jadhav  
**PRN:** 202301040033  
**Batch:** T1  
**Branch:** Computer Engineering  
**Subject:** Deep Learning / NLP Lab  

## Problem Statement
Implement NLP preprocessing techniques and build a text classification model using machine learning.

## Project Chosen for Submission
**Student Support Ticket Classification**

In this notebook, text queries raised by students are classified into one of the following categories:

- **Academic**
- **Fees**
- **Hostel**
- **Placement**

This topic is different from the reference notebook and uses a richer multi-class dataset with stronger model comparison and analysis.



## 1. Import Required Libraries

This section imports all required libraries for:
- data handling
- NLP preprocessing
- vectorization
- model training
- evaluation
- visualization


In [ ]:

import re
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully.")



## 2. NLP Tools Setup

The notebook first tries to use **NLTK** for tokenization, stopword removal, stemming, and lemmatization.

If NLTK resources are not available on the system, it automatically uses:
- regex tokenization
- Scikit-learn English stopwords
- Porter stemming
- a lightweight rule-based lemmatization fallback

This keeps the notebook runnable in restricted environments while still demonstrating all required preprocessing stages.


In [ ]:

# Safe NLP setup
USE_NLTK = True

try:
    import nltk
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer, WordNetLemmatizer

    def ensure_nltk_resource(resource_path, download_name):
        try:
            nltk.data.find(resource_path)
            return True
        except LookupError:
            try:
                nltk.download(download_name, quiet=True)
                nltk.data.find(resource_path)
                return True
            except Exception:
                return False

    punkt_ok = ensure_nltk_resource("tokenizers/punkt", "punkt")
    stop_ok = ensure_nltk_resource("corpora/stopwords", "stopwords")
    wordnet_ok = ensure_nltk_resource("corpora/wordnet", "wordnet")
    _ = ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")

    stemmer = PorterStemmer()
    lemmatizer = WordNetLemmatizer()

    if stop_ok:
        stop_words = set(stopwords.words("english"))
    else:
        stop_words = set(ENGLISH_STOP_WORDS)

    if not punkt_ok:
        USE_NLTK = False

except Exception:
    USE_NLTK = False
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
    stop_words = set(ENGLISH_STOP_WORDS)

def simple_tokenize(text):
    return re.findall(r"[a-zA-Z']+", text.lower())

def safe_tokenize(text):
    if USE_NLTK:
        try:
            return [tok.lower() for tok in word_tokenize(text) if re.match(r"[a-zA-Z']+$", tok)]
        except Exception:
            return simple_tokenize(text)
    return simple_tokenize(text)

def simple_lemmatize_word(word):
    rules = [
        ("ies", "y"),
        ("sses", "ss"),
        ("xes", "x"),
        ("ses", "s"),
        ("ing", ""),
        ("ed", ""),
        ("s", "")
    ]
    for old, new in rules:
        if word.endswith(old) and len(word) > len(old) + 2:
            candidate = word[:-len(old)] + new
            return candidate if len(candidate) > 2 else word
    return word

def safe_lemmatize(word):
    try:
        return lemmatizer.lemmatize(word)
    except Exception:
        return simple_lemmatize_word(word)

print("Using NLTK tokenization:", USE_NLTK)
print("Total stopwords used:", len(stop_words))
print("Stemmer:", stemmer.__class__.__name__)
print("Lemmatizer:", "WordNetLemmatizer" if 'lemmatizer' in globals() else "Rule-based lemmatizer")



## 3. Create and Explore Dataset

A richer synthetic dataset is generated for **student support ticket classification**.  
Each text sample is realistic and belongs to one of four classes:

- Academic
- Fees
- Hostel
- Placement

This makes the submission more original and more suitable for multi-class text classification.


In [ ]:

academic_templates = [
    "I am unable to understand the {topic} lecture and need extra notes before the {event}.",
    "Please help me with {topic} because the assignment deadline is near.",
    "I missed the {topic} class due to illness and want the recorded lecture.",
    "Can you share study material for {topic} and explain the important concepts?",
    "I need clarification on the {topic} practical file and internal marks.",
    "The faculty uploaded incomplete notes for {topic}; please update the module.",
    "I am confused about the syllabus of {topic} for the upcoming {event}.",
    "Kindly arrange a doubt session for {topic} before the university exam."
]

fees_templates = [
    "My {fee_type} fee payment is still pending on the portal even after transaction success.",
    "I need a receipt for my {fee_type} payment for scholarship verification.",
    "The payment gateway deducted money twice while paying the {fee_type} fees.",
    "Please tell me the last date and fine details for {fee_type} fee submission.",
    "I want to apply for installment option for the {fee_type} fees this semester.",
    "The portal is showing dues in my {fee_type} account though I already paid.",
    "Can the accounts section update my {fee_type} payment status before tomorrow?",
    "I need a bonafide and fee structure document for my {fee_type} loan process."
]

hostel_templates = [
    "There is a {issue} problem in my hostel room and it needs urgent attention.",
    "The hostel mess is serving {issue} food and many students are complaining.",
    "Please change my room because the {issue} situation is affecting my studies.",
    "I want to report a {issue} issue in the washroom on my hostel floor.",
    "The warden has not resolved the {issue} complaint raised last week.",
    "My room has {issue} and maintenance staff has not visited yet.",
    "Kindly solve the {issue} issue in block B as soon as possible.",
    "I need permission for temporary shift because of {issue} in my current room."
]

placement_templates = [
    "Please guide me about {topic} preparation for the upcoming company drive.",
    "I need help updating my resume for {topic} role applications.",
    "Can the placement cell share the eligibility criteria for {topic} companies?",
    "I missed the pre-placement talk and need details about the {topic} recruitment process.",
    "Please conduct a mock interview session focused on {topic} profiles.",
    "I want to know whether backlog students can sit for {topic} internships.",
    "Kindly share aptitude resources and previous questions for {topic} recruitment.",
    "I need clarification on the test pattern for {topic} campus placements."
]

academic_topics = ["data structures", "operating systems", "database management", "computer networks", "machine learning", "java programming", "cloud computing", "software engineering"]
academic_events = ["mid sem exam", "end sem exam", "viva", "practical exam", "unit test"]

fee_types = ["tuition", "exam", "hostel", "transport", "library", "semester", "development", "scholarship adjusted"]

hostel_issues = ["water leakage", "wifi", "electricity", "unclean", "mosquito", "fan not working", "bed damage", "plumbing"]

placement_topics = ["software developer", "data analyst", "java backend", "testing", "cloud engineer", "full stack", "internship", "graduate trainee"]

records = []

for _ in range(60):
    txt = random.choice(academic_templates).format(
        topic=random.choice(academic_topics),
        event=random.choice(academic_events)
    )
    txt += " " + random.choice([
        "Please respond quickly.",
        "This is important for my attendance and marks.",
        "I need support from the department.",
        "Kindly resolve this issue soon."
    ])
    records.append((txt, "Academic"))

for _ in range(60):
    txt = random.choice(fees_templates).format(
        fee_type=random.choice(fee_types)
    )
    txt += " " + random.choice([
        "My transaction ID is available.",
        "This is urgent because the deadline is close.",
        "Please verify from the accounts office.",
        "I do not want a late fee penalty."
    ])
    records.append((txt, "Fees"))

for _ in range(60):
    txt = random.choice(hostel_templates).format(
        issue=random.choice(hostel_issues)
    )
    txt += " " + random.choice([
        "Kindly send maintenance staff.",
        "This is causing discomfort every day.",
        "I already informed the warden once.",
        "Please solve it as early as possible."
    ])
    records.append((txt, "Hostel"))

for _ in range(60):
    txt = random.choice(placement_templates).format(
        topic=random.choice(placement_topics)
    )
    txt += " " + random.choice([
        "I want to improve my chances of selection.",
        "Please share details with the final year students.",
        "This would help before the aptitude round.",
        "I am very interested in participating."
    ])
    records.append((txt, "Placement"))

random.shuffle(records)

df = pd.DataFrame(records, columns=["text", "category"])
df.head(10)


In [ ]:

print("Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df["category"].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())


In [ ]:

ax = df["category"].value_counts().plot(kind="bar", figsize=(7,4), title="Class Distribution")
ax.set_xlabel("Category")
ax.set_ylabel("Number of Samples")
plt.tight_layout()
plt.show()



## 4. Text Preprocessing

The following preprocessing steps are implemented:

1. Lowercasing  
2. Tokenization  
3. Stopword removal  
4. Stemming  
5. Lemmatization  

A full preprocessing pipeline is then created for model training.


In [ ]:

sample_text = df.loc[0, "text"]
print("Original Text:\n", sample_text)

tokens = safe_tokenize(sample_text)
tokens_wo_stop = [w for w in tokens if w not in stop_words]
stemmed = [stemmer.stem(w) for w in tokens_wo_stop]
lemmatized = [safe_lemmatize(w) for w in tokens_wo_stop]

print("\nTokenized Words:\n", tokens)
print("\nAfter Stopword Removal:\n", tokens_wo_stop)
print("\nAfter Stemming:\n", stemmed)
print("\nAfter Lemmatization:\n", lemmatized)


In [ ]:

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    tokens = safe_tokenize(text)
    tokens = [tok for tok in tokens if tok not in stop_words and len(tok) > 2]
    stemmed_tokens = [stemmer.stem(tok) for tok in tokens]
    lemmatized_tokens = [safe_lemmatize(tok) for tok in stemmed_tokens]
    return " ".join(lemmatized_tokens)

df["clean_text"] = df["text"].apply(preprocess_text)

df[["text", "clean_text", "category"]].head(10)


In [ ]:

print("Sample Preprocessed Texts:")
for i in range(5):
    print(f"\nSample {i+1}")
    print("Original :", df.loc[i, "text"])
    print("Processed:", df.loc[i, "clean_text"])



## 5. Text Vectorization

Two vectorization techniques are used:

- **TF-IDF Vectorizer**
- **CountVectorizer**

These convert the cleaned text into numerical feature vectors.


In [ ]:

tfidf_vectorizer = TfidfVectorizer(
    max_features=250,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

count_vectorizer = CountVectorizer(
    max_features=250,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

X_tfidf = tfidf_vectorizer.fit_transform(df["clean_text"])
X_count = count_vectorizer.fit_transform(df["clean_text"])
y = df["category"]

print("TF-IDF Shape :", X_tfidf.shape)
print("Count Shape  :", X_count.shape)


In [ ]:

tfidf_features = tfidf_vectorizer.get_feature_names_out()[:20]
count_features = count_vectorizer.get_feature_names_out()[:20]

print("Top TF-IDF Features:")
print(tfidf_features)

print("\nTop CountVectorizer Features:")
print(count_features)



## 6. Train-Test Split

The dataset is split into:
- **80% Training**
- **20% Testing**

A **stratified split** is used to preserve class balance.


In [ ]:

X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

X_train_count, X_test_count, _, _ = train_test_split(
    X_count, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train_tfidf.shape[0])
print("Testing samples :", X_test_tfidf.shape[0])



## 7. Build Classification Models

The following models are trained:

1. **Multinomial Naive Bayes**
2. **Logistic Regression**
3. **Linear SVM**
4. **MLP Classifier (Neural Network based model)**

This provides stronger comparison and satisfies both machine learning and model evaluation requirements.


In [ ]:

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Linear SVM": LinearSVC(),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(40,), max_iter=200, random_state=42)
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully.")



## 8. Evaluate Model Performance

Each model is evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Classification Report
- Confusion Matrix
- 5-fold Cross Validation Accuracy


In [ ]:

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro")
    cv_value = 3 if name == "MLP Classifier" else 5
    cv_scores = cross_val_score(model, X_tfidf, y, cv=cv_value, scoring="accuracy")

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "CV Mean Accuracy": round(cv_scores.mean(), 4)
    })

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
results_df


In [ ]:

best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]
print("Best Model Selected:", best_model_name)


In [ ]:

for name, model in trained_models.items():
    print("\n" + "="*90)
    print(f"{name} - Classification Report")
    print("="*90)
    y_pred = model.predict(X_test_tfidf)
    print(classification_report(y_test, y_pred))


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score", "CV Mean Accuracy"]]
plot_df.plot(kind="bar", ax=ax)
ax.set_title("Model Comparison on TF-IDF Features")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:

for name, model in trained_models.items():
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred, labels=sorted(df["category"].unique()))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(df["category"].unique()))
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.show()



## 9. TF-IDF vs CountVectorizer Comparison on Best Models

To compare vectorization methods more clearly, Logistic Regression and Naive Bayes are trained on both feature representations.


In [ ]:

comparison_rows = []

vector_sets = {
    "TF-IDF": (X_train_tfidf, X_test_tfidf),
    "CountVectorizer": (X_train_count, X_test_count)
}

for vec_name, (xtr, xte) in vector_sets.items():
    for name, model in {
        "Naive Bayes": MultinomialNB(),
        "Logistic Regression": LogisticRegression(max_iter=2000)
    }.items():
        model.fit(xtr, y_train)
        preds = model.predict(xte)
        acc = accuracy_score(y_test, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="macro")
        comparison_rows.append({
            "Vectorizer": vec_name,
            "Model": name,
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1-Score": round(f1, 4)
        })

vec_compare_df = pd.DataFrame(comparison_rows).sort_values(by=["F1-Score", "Accuracy"], ascending=False)
vec_compare_df



## 10. Predict on New Unseen Student Queries

The best model is tested on new manually created examples.


In [ ]:

new_queries = [
    "The accounts portal still shows unpaid tuition fees even after online payment.",
    "Please arrange mock interview practice for cloud engineer campus recruitment.",
    "There is water leakage in my hostel room and the washroom is very dirty.",
    "I need notes for operating systems because I missed the lecture and exam is near."
]

new_queries_clean = [preprocess_text(q) for q in new_queries]
new_vectors = tfidf_vectorizer.transform(new_queries_clean)
predictions = best_model.predict(new_vectors)

prediction_df = pd.DataFrame({
    "Student Query": new_queries,
    "Predicted Category": predictions
})

prediction_df



## 11. Analysis of the Implemented Models

### Observations
- All models performed strongly because the dataset is clean and category-specific.
- **TF-IDF** performed better than raw count vectors in most cases because it gives more importance to informative terms and reduces the effect of highly frequent common words.
- **Linear SVM** and **Logistic Regression** usually perform better on text classification because they handle sparse high-dimensional vectors very effectively.
- **Naive Bayes** is fast and simple, but it assumes feature independence, which is often an oversimplification.
- **MLP Classifier** adds a neural-network based comparison, but classical linear models can still outperform it on sparse text data.

### Why the Best Model Worked Well
The best model performed well because:
- preprocessing reduced noise
- TF-IDF captured important terms
- the class labels were semantically distinct
- the data distribution was balanced

### Alternative Models That Could Also Be Used
- Random Forest
- XGBoost
- LSTM / GRU based deep learning models
- BERT or transformer-based models for advanced NLP

### Real-World Use Cases
This type of model can be used for:
- automatic complaint routing in college ERP systems
- student helpdesk categorization
- customer support email classification
- service desk ticket automation



## 12. Conclusion

This assignment successfully implemented:

- Tokenization
- Stopword removal
- Stemming
- Lemmatization
- TF-IDF vectorization
- CountVectorizer
- Text classification using multiple models
- Performance evaluation using standard metrics

### Final Conclusion
The notebook demonstrates a complete NLP pipeline from raw text to classification and evaluation.  
Among all tested models, the best-performing model can be selected for practical deployment in a student support ticket routing system.



## GitHub Repository Link
Upload the project files to your GitHub repository and paste the final link here before submission.

**GitHub Link:** `Add your GitHub repository link here`
